<img src="../shared/alchemi-banner-left.png" alt="NVIDIA ALCHEMI: AI for Chemistry and Materials Science" style="display:block;box-sizing:border-box;width:100%;max-width:100%;height:auto;">

<img src="assets/agent02-top-light.svg" alt="Three structures of different sizes become AtomicData, stack into one Batch, go to the GPU in one model call, return energy and forces for each structure, and are saved to a Zarr store." style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0;">

<div style="border: 1px solid rgba(127,127,127,0.35); border-left: 5px solid #76b900; border-radius: 4px; padding: 24px 28px; margin: 8px 0 18px; background: transparent; color: inherit; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif;"><div style="font-variant: small-caps; font-size: 12px; font-weight: 700; letter-spacing: 2px; opacity: 0.75;">AGENTIC TRACK, NOTEBOOK 02</div><h1 style="margin: 6px 0 8px; padding: 0; border: 0;"><span style="font-size: 32px; font-weight: 700; line-height: 1.2; color: inherit;">Many molecules, one batch</span></h1><div style="font-size: 18px; line-height: 1.45; opacity: 0.85;">How to describe structures for Toolkit, pack them into one batch, and save what you compute.</div></div>

<div style="border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 12px 18px; margin: 12px 0 18px; background: transparent; color: inherit; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; line-height: 1.6;"><b>Hardware:</b> a CUDA-capable NVIDIA GPU<br><b>Prerequisites:</b> Meet your agent<br><b>References:</b> <a href="https://nvidia.github.io/nvalchemi-toolkit/" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">Toolkit documentation</a>; <a href="https://nvidia.github.io/nvalchemi-toolkit/examples/" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">Toolkit examples</a>; <a href="https://github.com/NVIDIA/skills/tree/main/skills/nvalchemi-toolkit" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">the nvalchemi-toolkit skill</a></div>

<img src="assets/agent02-roadmap-light.svg" alt="Sections of this notebook on one line." style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0;">

Your structures live in files and in other tools. Toolkit works on tensors
on the GPU. This lesson turns one into the other, runs one model call on
all of them, and saves the results. You will meet these classes along the
way:

- `AtomicData`: one structure, as tensors
- `Batch`: many structures packed together
- `AtomicDataZarrWriter` and `AtomicDataZarrReader`: save to disk and read
  back

You will drive your agent through each step.

<img src="assets/agent02-packed-rows-light.svg" alt="Atoms of ammonia, propyne and phenol packed as 24 rows in one Toolkit batch; a padded layout of the same three needs 39 slots." style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0;">

In [ ]:
from helpers import agent02 as helpers   # course loaders, tables, plots
helpers.configure_tutorial()             # course display defaults
work_dir = helpers.make_work_dir()       # notebooks/agent02_work
n_structures = 1000                      # structures in the timing batch
helpers.show_session()                   # prints what your agent needs to start

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">1. One structure as AtomicData</h2>

A structure file lists atoms and coordinates. A model needs tensors on the
GPU. AtomicData is Toolkit's form for one structure: every field has a
name, a shape and a unit, and each one belongs either to the atoms or to
the structure as a whole.

Write the prompt for this step.

| | |
| --- | --- |
| input  | `helpers.load_molecule("caffeine")` |
| output | `caffeine_data`, in `agent02_work/atomic_data.py` |

In [ ]:
%load agent02_work/atomic_data.py

In [ ]:
# Show the fields of caffeine_data
helpers.show_fields(caffeine_data)

Each row is one field: its shape tells you whether it holds one value per
atom or one for the whole molecule. The tensors stay on the CPU until you
name a device.

> ⚙️ **Fine control**
>
> Your agent proposes the settings. To choose one yourself, look at the
> parameters and name the exact value in your prompt.
>
> ```python
> AtomicData.from_atoms(
>     atoms,          # an ASE Atoms object
>     device=...,     # where the tensors are created
> )
> ```
>
> Example: `build it in float64`

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">2. Many structures in one Batch</h2>

A model call on one molecule leaves most of the GPU idle. Toolkit packs
many structures into one Batch: their atoms are stacked into shared
tensors, and an index records which molecule each atom belongs to. A
densely packed batch gives you the best throughput for your GPU budget.

Write the prompt for this step.

| | |
| --- | --- |
| input  | `helpers.load_molecule(name)` for caffeine, aspirin, ibuprofen, paracetamol, capsaicin |
| output | `molecule_data`, `batch`, in `agent02_work/batch.py` |

In [ ]:
%load agent02_work/batch.py

In [ ]:
# Show which atom rows belong to which molecule
helpers.show_batch_rows(batch)

Each colour is one molecule; the index under the strip says which molecule
every atom row belongs to.

> ⚙️ **Fine control**
>
> ```python
> Batch.from_data_list(
>     data_list,      # AtomicData objects, in the order you want
>     device=...,     # where the batch lives
> )
> ```
>
> Example: `put ibuprofen first`

#### Try it: add a molecule of your own

Bring a molecule from your own work, as a structure file or a SMILES
string. Ask your agent to put it into the batch at a position you choose.
Then print the atom counts of the batch and check that yours sits where you
asked.

| | |
| --- | --- |
| input  | your file or SMILES; `molecule_data` |
| output | `my_batch`, in `agent02_work/my_batch.py` |

Done when: the count at your position is your molecule's atom count.

> 💡 **Hint**
>
> Nothing at hand? `helpers.load_default_molecule()` gives you a tested default.

In [ ]:
%load agent02_work/my_batch.py

In [ ]:
print(my_batch.num_nodes_per_graph)

#### Try it: CPU first, or straight to the GPU

Structures usually arrive as ASE or pymatgen objects, and for `n_structures` of them the way the batch is built shows in the time. Ask your agent to build one batch of that size three ways and to time each: from the objects on the CPU and then moved to the GPU, from the objects directly on the GPU, and directly from tensors. Time the build only, once the coordinates exist.

| | |
| --- | --- |
| input  | `helpers.load_many(n_structures)` |
| output | `build_table`, in `agent02_work/build_routes.py` |

Done when: the table gives a time for each of the three routes.

In [ ]:
%load agent02_work/build_routes.py

In [ ]:
# Show the build time of each route
helpers.show_build_routes(build_table)

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">3. One model call on the batch</h2>

Back to `batch`, your five molecules: one call to the model handles every
molecule in it. The model returns an energy for each structure and a force
on each atom, as tensors laid out like the batch itself.

This call uses AIMNet2 as Toolkit ships it. A later lesson adds the interaction terms it needs for accurate results.

Write the prompt for this step.

| | |
| --- | --- |
| input  | `batch`; AIMNet2, built-in wrapper |
| output | `aimnet_model`, `energy`, `forces`, in `agent02_work/model_call.py` |

In [ ]:
%load agent02_work/model_call.py

In [ ]:
# Show the energy and the largest force of each molecule
helpers.show_results(batch, energy, forces)

In [ ]:
print(energy.shape, forces.shape)

| output | shape | one row per | unit |
| --- | --- | --- | --- |
| `energy` | (5, 1) | structure | eV |
| `forces` | (147, 3) | atom | eV/Å |

`batch_idx` links each atom row to its structure.

> </> API
>
> `AIMNet2Wrapper.from_checkpoint(path, device=...)`
>
> **Input:** a checkpoint and the device<br>
> **Effect:** loads the pretrained AIMNet2; Coulomb and D3 dispersion are not included

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">4. Save the results with Zarr</h2>

Results that live only in GPU memory are gone when the session ends.
Toolkit writes a batch to a Zarr store on disk: the structures and what you
computed for them, with the same field names and shapes. The store can be
read back later, extended, or used for training.

Write the prompt for this step.

| | |
| --- | --- |
| input  | `batch`; `energy`; `forces` |
| output | `store_path`, `loaded_batch`, in `agent02_work/zarr_roundtrip.py` |

In [ ]:
%load agent02_work/zarr_roundtrip.py

In [ ]:
# Show what the store holds and compare it with what you saved
helpers.show_store(store_path, batch, loaded_batch)

The table lists what the store holds, and its last line confirms that what
you read back equals what you saved.

> ⚙️ **Fine control**
>
> ```python
> AtomicDataZarrWriter(path).write(batch)    # a new store
> AtomicDataZarrWriter(path).append(batch)   # add to an existing one
> AtomicDataZarrReader(path)                 # read it back
> ```
>
> Example: `append to the store instead of writing a new one`

> </> API
>
> `Dataset(AtomicDataZarrReader(path), device=...)`
>
> **Input:** a reader on the store<br>
> **Effect:** each sample comes back as `AtomicData` on the device

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">Explore more</h2>

More of Toolkit's data tools, each with its official page.

<div style="display: flex; flex-wrap: wrap; gap: 16px; margin: 4px 0 18px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif;"><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5; overflow-wrap: break-word;"><img src="assets/explore-workflows-light.svg" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div><b>Batches from a store</b></div><div style="margin: 2px 0 0;"><a href="https://github.com/NVIDIA/nvalchemi-toolkit/blob/v0.3.0/docs/userguide/datapipes.md">Docs</a>, <a href="https://github.com/NVIDIA/nvalchemi-toolkit/blob/v0.3.0/examples/intermediate/02_trajectory_zarr_io.py">Example</a></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8;"><code>DataLoader</code> reads a store back in batches.</div></div><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5; overflow-wrap: break-word;"><img src="assets/explore-workflows-light.svg" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div><b>Fields of your own</b></div><div style="margin: 2px 0 0;"><a href="https://github.com/NVIDIA/nvalchemi-toolkit/blob/v0.3.0/docs/userguide/data.md">Docs</a>, <a href="https://github.com/NVIDIA/nvalchemi-toolkit/blob/v0.3.0/examples/basic/01_data_structures.py">Example</a></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8;">Attach your own per-atom data to a batch.</div></div><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5; overflow-wrap: break-word;"><img src="assets/explore-workflows-light.svg" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div><b>Chunks and compression</b></div><div style="margin: 2px 0 0;"><a href="https://github.com/NVIDIA/nvalchemi-toolkit/blob/v0.3.0/docs/userguide/zarr_compression.md">Docs</a></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8;">Settings that decide how fast a large store reads.</div></div><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5; overflow-wrap: break-word;"><img src="assets/explore-workflows-light.svg" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div><b>Snapshots during a run</b></div><div style="margin: 2px 0 0;"><a href="https://github.com/NVIDIA/nvalchemi-toolkit/blob/v0.3.0/docs/userguide/dynamics_sinks.md">Docs</a>, <a href="https://github.com/NVIDIA/nvalchemi-toolkit/blob/v0.3.0/examples/intermediate/02_trajectory_zarr_io.py">Example</a></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8;">A simulation writes its frames to a store.</div></div></div>

<div style="border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; background: transparent; color: inherit; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; padding: 15px 20px; margin: 24px 0 18px; line-height: 1.6;"><h2 style="margin: 0 0 10px; padding: 0; border: 0; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 24px; font-weight: 500; line-height: 1.25; color: inherit;">Notebook 02 recap</h2><div style="margin: 0 0 10px;">You described molecules as AtomicData, packed them into one Batch, ran one model call on all of them, and saved the results to a Zarr store. Every quantity kept its name, shape and unit along the way.</div><div style="margin: 0 0 10px;">The next lessons put batches to work: relaxation and molecular dynamics with built-in models, then models you compose and wrap yourself.</div><div style="margin: 0;"><img src="assets/fork-agent-light.svg" alt="" width="18" height="18" style="width: 18px; height: 18px; vertical-align: -3px; margin-right: 6px;"><b>Next:</b> <a href="alchemi-agent-03-ready-made.ipynb" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">Ready-made →</a></div></div>